# Backlog Agent — user perspective

**What it does.** Reads a free-form `plan.md` and creates the matching Linear `EPIC -> User Story -> Task` tree. Idempotent: re-running with the same plan reuses existing EPICs (BKPK-05).

**Entry point.** `run_backlog_agent(input: BacklogInput) -> BacklogOutput` (sync wrapper around an async loop).

**Cost.** Live runs hit Linear MCP and consume tokens. Default state of this notebook = construct the input + show what would be sent. Set `HSB_NOTEBOOK_RUN_LIVE=1` plus `HSB_NOTEBOOK_PLAN_MD=/path/to/plan.md` to actually invoke.

**Runtime.** Backlog is the canary that goes through `make_agent_options()` + `resolve_runtime("backlog")`, so `HSB_RUNTIME_BACKLOG=codex` works end-to-end.

In [ ]:
import sys
from pathlib import Path

# Robust _helpers.py discovery — works whether jupyter lab was launched from
# the repo root, notebooks/, or notebooks/agents/. The loop walks up from cwd
# until it finds the parent that holds _helpers.py and inserts it on sys.path.
nb_dir = Path.cwd()
for _parent in (nb_dir, *nb_dir.parents):
    if (_parent / "_helpers.py").exists():
        sys.path.insert(0, str(_parent))
        break

from _helpers import ensure_src_on_path, runtime_summary

ROOT = ensure_src_on_path()
print("HSB_RUNTIME_<AGENT> selection:\n" + runtime_summary())

## Construct the input

`BacklogInput` requires `plan_source` (absolute path to plan.md) and a `ProjectContext`. Both models forbid extra fields — typos surface as `ValidationError` at construction time.

In [ ]:
from hsb.contracts.backlog import BacklogInput, ProjectContext

example_input = BacklogInput(
    plan_source="docs/plan.md",
    project_context=ProjectContext(
        name="hsb-demo",
        repository="task-management-agents",
        technical_stack=["python", "claude-agent-sdk"],
    ),
)
print(example_input.model_dump_json(indent=2))

## Live invocation

Gates: `HSB_NOTEBOOK_RUN_LIVE=1` AND `HSB_NOTEBOOK_PLAN_MD=/abs/path/to/plan.md`. The second call demonstrates idempotency — re-invoking with the same plan should not change the EPIC count.

In [ ]:
import os

from _helpers import assert_g1_safe, gated, live_mode, selected_runtime

plan_md = os.environ.get("HSB_NOTEBOOK_PLAN_MD")
if not live_mode() or not plan_md:
    print(gated("Backlog live run (set HSB_NOTEBOOK_PLAN_MD)"))
else:
    assert_g1_safe()
    print(f"(running on HSB_RUNTIME_BACKLOG={selected_runtime('backlog')!r})")
    from hsb.agents.backlog_agent import run_backlog_agent

    inp = BacklogInput(
        plan_source=plan_md,
        project_context=ProjectContext(
            name="hsb-demo",
            repository="task-management-agents",
            technical_stack=["python"],
        ),
    )
    out = run_backlog_agent(inp)
    print(f"EPICs created: {len(out.epics)}")
    for e in out.epics[:3]:
        print(f"  - {e.title}  ({len(e.user_stories)} stories)")

    # BKPK-05 idempotency: rerun should reuse existing EPICs.
    out2 = run_backlog_agent(inp)
    print(
        "rerun EPIC count:",
        len(out2.epics),
        "(same)" if len(out2.epics) == len(out.epics) else "(DIFFERENT — investigate)",
    )